In [1]:
from __future__ import annotations

import json
import random
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path

import joblib
import folium
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from IPython.display import display


# ============================================================
# CONFIG
# ============================================================

@dataclass
class Config:
    train_path: str = "train_data.csv"
    test_path: str = "test_data.csv"

    seq_len: int = 24
    batch_size: int = 256
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    dropout: float = 0.25
    kernel_size: int = 5
    tcn_channels: tuple = (64, 64, 128, 256)
    dilations: tuple = (1, 2, 4, 8)

    max_epochs: int = 100
    min_epochs: int = 10
    loss_stability_threshold: float = 0.01

    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    track_id_col: str = "voyage_id"
    vessel_id_col: str = "MMSI"
    row_id_col: str = "row_id"
    time_col: str = "TIME"
    lat_col: str = "LAT"
    lon_col: str = "LON"
    speed_col: str = "SPEED"
    cog_col: str = "COG"
    heading_col: str = "HEADING"
    dt_col: str = "dt"
    num_pings_col: str = "num_pings"

    earth_radius_m: float = 6_371_000.0
    knots_to_mps: float = 0.514444
    meters_to_yards: float = 1.0936132983377078

    dr_horizon_mode: str = "input_dt"   # "input_dt" or "target_dt"


CFG = Config()
BASE_DIR = Path.cwd()
ARTIFACTS_DIR = BASE_DIR / "latest_tcn_run"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# UTILITIES
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def haversine_m(lat1, lon1, lat2, lon2, R=6_371_000.0):
    lat1 = np.deg2rad(lat1)
    lon1 = np.deg2rad(lon1)
    lat2 = np.deg2rad(lat2)
    lon2 = np.deg2rad(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2.0 * np.arcsin(np.sqrt(a))
    return R * c


def summarize_error(error_yds: np.ndarray) -> dict:
    return {
        "rmse_yds": float(np.sqrt(np.mean(error_yds ** 2))),
        "mae_yds": float(np.mean(error_yds)),
        "median_yds": float(np.median(error_yds)),
        "p95_yds": float(np.percentile(error_yds, 95)),
    }


def within_2_percent(a, b, threshold=0.02):
    denom = max((a + b) / 2, 1e-12)
    return abs(a - b) / denom <= threshold


# ============================================================
# DATA PREP
# ============================================================

def load_data(path: str, cfg: Config) -> pd.DataFrame:
    df = pd.read_csv(path)
    df[cfg.time_col] = pd.to_datetime(df[cfg.time_col], errors="coerce")

    numeric_cols = [
        cfg.lat_col, cfg.lon_col, cfg.speed_col,
        cfg.cog_col, cfg.heading_col, cfg.dt_col
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df[cfg.dt_col] = df[cfg.dt_col].fillna(0.0)
    df.loc[df[cfg.dt_col] < 0, cfg.dt_col] = 0.0
    df.loc[df[cfg.dt_col] > 1800, cfg.dt_col] = 0.0
    df[cfg.heading_col] = df[cfg.heading_col].fillna(df[cfg.cog_col])

    required = [
        cfg.track_id_col, cfg.vessel_id_col, cfg.row_id_col, cfg.num_pings_col,
        cfg.time_col, cfg.lat_col, cfg.lon_col, cfg.speed_col,
        cfg.cog_col, cfg.heading_col, cfg.dt_col
    ]
    df = df.dropna(subset=required).copy()

    df[cfg.vessel_id_col] = df[cfg.vessel_id_col].astype(str)
    df[cfg.track_id_col] = df[cfg.track_id_col].astype(str)
    df[cfg.row_id_col] = df[cfg.row_id_col].astype(str)

    df = df.sort_values([cfg.track_id_col, cfg.time_col]).reset_index(drop=True)
    return df


def add_features_and_targets(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    df = df.copy()

    df["COG_rad"] = np.deg2rad(df[cfg.cog_col] % 360.0)
    df["COG_cos"] = np.cos(df["COG_rad"])
    df["COG_sin"] = np.sin(df["COG_rad"])

    df["HEADING_rad"] = np.deg2rad(df[cfg.heading_col] % 360.0)
    df["HEADING_cos"] = np.cos(df["HEADING_rad"])
    df["HEADING_sin"] = np.sin(df["HEADING_rad"])

    df["next_lat"] = df.groupby(cfg.track_id_col)[cfg.lat_col].shift(-1)
    df["next_lon"] = df.groupby(cfg.track_id_col)[cfg.lon_col].shift(-1)
    df["target_dt"] = df.groupby(cfg.track_id_col)[cfg.dt_col].shift(-1)
    df["next_time"] = df.groupby(cfg.track_id_col)[cfg.time_col].shift(-1)

    df["dlat"] = df["next_lat"] - df[cfg.lat_col]
    df["dlon"] = df["next_lon"] - df[cfg.lon_col]

    df = df.dropna(
        subset=["dlat", "dlon", "next_lat", "next_lon", "target_dt", "next_time"]
    ).reset_index(drop=True)

    return df


FEATURE_COLS = [
    "LAT", "LON", "SPEED", "dt",
    "COG_cos", "COG_sin", "HEADING_cos", "HEADING_sin"
]
TARGET_COLS = ["dlat", "dlon"]


def build_sequences(df: pd.DataFrame, cfg: Config):
    X_list, y_list, last_pos_list, meta_rows = [], [], [], []

    for _, group in df.groupby(cfg.track_id_col, sort=False):
        group = group.sort_values(cfg.time_col).reset_index(drop=True)

        if len(group) <= cfg.seq_len:
            continue

        X_vals = group[FEATURE_COLS].to_numpy(dtype=np.float32)
        y_vals = group[TARGET_COLS].to_numpy(dtype=np.float32)
        latlon_vals = group[[cfg.lat_col, cfg.lon_col]].to_numpy(dtype=np.float32)

        for i in range(len(group) - cfg.seq_len):
            last_idx = i + cfg.seq_len - 1
            pred_idx = i + cfg.seq_len

            X_seq = X_vals[i:i + cfg.seq_len]
            y_next = y_vals[last_idx]
            last_pos = latlon_vals[last_idx]

            X_list.append(X_seq)
            y_list.append(y_next)
            last_pos_list.append(last_pos)

            meta_rows.append({
                "row_id": group.loc[pred_idx, cfg.row_id_col],
                "MMSI": group.loc[pred_idx, cfg.vessel_id_col],
                "voyage_id": group.loc[pred_idx, cfg.track_id_col],
                "anchor_time": group.loc[last_idx, cfg.time_col],
                "pred_time": group.loc[pred_idx, cfg.time_col],
                "input_dt": float(group.loc[last_idx, cfg.dt_col]),
                "target_dt": float(group.loc[pred_idx, cfg.dt_col]),
                "num_pings": group.loc[pred_idx, cfg.num_pings_col],
                "last_speed": float(group.loc[last_idx, cfg.speed_col]),
                "last_cog": float(group.loc[last_idx, cfg.cog_col]),
                "last_heading": float(group.loc[last_idx, cfg.heading_col]),
            })

    if not X_list:
        raise ValueError("No sequences built. Check seq_len or input files.")

    X = np.stack(X_list).astype(np.float32)
    y = np.stack(y_list).astype(np.float32)
    last_pos = np.stack(last_pos_list).astype(np.float32)
    meta_df = pd.DataFrame(meta_rows)

    return X, y, last_pos, meta_df


def scale_X_train_test(X_train: np.ndarray, X_test: np.ndarray):
    n_train, seq_len, n_features = X_train.shape
    n_test = X_test.shape[0]

    scaler = StandardScaler()
    scaler.fit(X_train.reshape(n_train * seq_len, n_features))

    X_train_scaled = scaler.transform(
        X_train.reshape(n_train * seq_len, n_features)
    ).reshape(n_train, seq_len, n_features)

    X_test_scaled = scaler.transform(
        X_test.reshape(n_test * seq_len, n_features)
    ).reshape(n_test, seq_len, n_features)

    return X_train_scaled.astype(np.float32), X_test_scaled.astype(np.float32), scaler


# ============================================================
# DATASET / MODEL
# ============================================================

class SequenceDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class Chomp1d(nn.Module):
    def __init__(self, chomp_size: int):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous() if self.chomp_size > 0 else x


class TemporalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout):
        super().__init__()
        padding = (kernel_size - 1) * dilation

        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size, padding=padding, dilation=dilation),
            Chomp1d(padding),
            nn.ELU(),
            nn.Dropout(dropout),

            nn.Conv1d(out_ch, out_ch, kernel_size, padding=padding, dilation=dilation),
            Chomp1d(padding),
            nn.ELU(),
            nn.Dropout(dropout),
        )

        self.downsample = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None
        self.final_act = nn.ELU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.final_act(out + res)


class TCN(nn.Module):
    def __init__(self, input_dim, output_dim, channels, kernel_size, dilations, dropout):
        super().__init__()
        layers = []
        in_ch = input_dim

        for out_ch, dilation in zip(channels, dilations):
            layers.append(TemporalBlock(in_ch, out_ch, kernel_size, dilation, dropout))
            in_ch = out_ch

        self.tcn = nn.Sequential(*layers)
        self.fc = nn.Linear(in_ch, output_dim)

    def forward(self, x):
        x = x.transpose(1, 2)
        z = self.tcn(x)
        z_last = z[:, :, -1]
        return self.fc(z_last)


# ============================================================
# TRAIN / PREDICT / DR
# ============================================================

def train_model(model, train_loader, cfg: Config):
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )
    loss_fn = nn.MSELoss()

    train_losses = []

    for epoch in range(cfg.max_epochs):
        model.train()
        total_loss = 0.0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(cfg.device)
            y_batch = y_batch.to(cfg.device)

            optimizer.zero_grad()
            preds = model(X_batch)
            loss = loss_fn(preds, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item() * X_batch.size(0)

        avg_loss = total_loss / len(train_loader.dataset)
        train_losses.append(avg_loss)
        print(f"Epoch {epoch+1:03d}/{cfg.max_epochs} | train_loss={avg_loss:.8f}")

        if len(train_losses) >= 3 and (epoch + 1) >= cfg.min_epochs:
            l1, l2, l3 = train_losses[-3:]
            if (
                within_2_percent(l1, l2, cfg.loss_stability_threshold)
                and within_2_percent(l2, l3, cfg.loss_stability_threshold)
                and within_2_percent(l1, l3, cfg.loss_stability_threshold)
            ):
                print(
                    f"\nStopping early after epoch {epoch+1}: "
                    f"last 3 losses are all within {cfg.loss_stability_threshold * 100:.1f}%."
                )
                break

    return train_losses


def predict_model(model, loader, cfg: Config):
    model.eval()
    preds = []

    with torch.no_grad():
        for X_batch, _ in loader:
            X_batch = X_batch.to(cfg.device)
            preds.append(model(X_batch).cpu().numpy())

    return np.vstack(preds)


def reconstruct_latlon(last_positions: np.ndarray, deltas: np.ndarray):
    pred_lat = last_positions[:, 0] + deltas[:, 0]
    pred_lon = last_positions[:, 1] + deltas[:, 1]
    return np.column_stack([pred_lat, pred_lon]).astype(np.float32)


def forward_dead_reckoning(
    lat_deg: np.ndarray,
    lon_deg: np.ndarray,
    speed_knots: np.ndarray,
    cog_deg: np.ndarray,
    dt_seconds: np.ndarray,
    R: float,
    knots_to_mps: float,
) -> np.ndarray:
    lat1 = np.deg2rad(lat_deg)
    lon1 = np.deg2rad(lon_deg)
    brng = np.deg2rad(cog_deg % 360.0)
    d = np.maximum(dt_seconds, 0.0) * np.maximum(speed_knots, 0.0) * knots_to_mps
    ang_dist = d / R

    sin_lat1 = np.sin(lat1)
    cos_lat1 = np.cos(lat1)
    sin_ang = np.sin(ang_dist)
    cos_ang = np.cos(ang_dist)

    lat2 = np.arcsin(sin_lat1 * cos_ang + cos_lat1 * sin_ang * np.cos(brng))
    lon2 = lon1 + np.arctan2(
        np.sin(brng) * sin_ang * cos_lat1,
        cos_ang - sin_lat1 * np.sin(lat2),
    )

    lon2 = (lon2 + np.pi) % (2 * np.pi) - np.pi
    return np.column_stack([np.rad2deg(lat2), np.rad2deg(lon2)]).astype(np.float32)


def choose_dr_horizon(meta_df: pd.DataFrame, cfg: Config) -> np.ndarray:
    if cfg.dr_horizon_mode == "target_dt":
        return meta_df["target_dt"].to_numpy(dtype=np.float32)
    return meta_df["input_dt"].to_numpy(dtype=np.float32)


# ============================================================
# RUN EVERYTHING
# ============================================================

set_seed(CFG.seed)

train_df_raw = load_data(CFG.train_path, CFG)
test_df_raw = load_data(CFG.test_path, CFG)

train_df = add_features_and_targets(train_df_raw, CFG)
test_df = add_features_and_targets(test_df_raw, CFG)

X_train, y_train, last_pos_train, meta_train = build_sequences(train_df, CFG)
X_test, y_test, last_pos_test, meta_test = build_sequences(test_df, CFG)

X_test_unscaled = X_test.copy()
X_train_scaled, X_test_scaled, feature_scaler = scale_X_train_test(X_train, X_test)

train_ds = SequenceDataset(X_train_scaled, y_train)
test_ds = SequenceDataset(X_test_scaled, y_test)

train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=CFG.batch_size, shuffle=False)

model = TCN(
    input_dim=X_train_scaled.shape[-1],
    output_dim=y_train.shape[-1],
    channels=CFG.tcn_channels,
    kernel_size=CFG.kernel_size,
    dilations=CFG.dilations,
    dropout=CFG.dropout,
).to(CFG.device)

print(f"Device: {CFG.device}")
train_losses = train_model(model, train_loader, CFG)

pred_test = predict_model(model, test_loader, CFG)

tcn_pred_latlon = reconstruct_latlon(last_pos_test, pred_test)
true_latlon = reconstruct_latlon(last_pos_test, y_test)

tcn_error_m = haversine_m(
    tcn_pred_latlon[:, 0], tcn_pred_latlon[:, 1],
    true_latlon[:, 0], true_latlon[:, 1],
    CFG.earth_radius_m
)
tcn_error_yds = tcn_error_m * CFG.meters_to_yards

dr_dt = choose_dr_horizon(meta_test, CFG)
dr_pred_latlon = forward_dead_reckoning(
    lat_deg=last_pos_test[:, 0],
    lon_deg=last_pos_test[:, 1],
    speed_knots=meta_test["last_speed"].to_numpy(dtype=np.float32),
    cog_deg=meta_test["last_cog"].to_numpy(dtype=np.float32),
    dt_seconds=dr_dt,
    R=CFG.earth_radius_m,
    knots_to_mps=CFG.knots_to_mps
)
dr_error_m = haversine_m(
    dr_pred_latlon[:, 0], dr_pred_latlon[:, 1],
    true_latlon[:, 0], true_latlon[:, 1],
    CFG.earth_radius_m
)
dr_error_yds = dr_error_m * CFG.meters_to_yards

dt_idx = FEATURE_COLS.index("dt")
results_rows = []

for i in range(len(pred_test)):
    tcn_err = float(tcn_error_yds[i])
    dr_err = float(dr_error_yds[i])
    improvement = dr_err - tcn_err
    winner = "TCN" if tcn_err < dr_err else ("DR" if dr_err < tcn_err else "TIE")

    results_rows.append({
        "idx": i,
        "row_id": meta_test.iloc[i]["row_id"],
        "MMSI": meta_test.iloc[i]["MMSI"],
        "voyage_id": meta_test.iloc[i]["voyage_id"],
        "anchor_time": meta_test.iloc[i]["anchor_time"],
        "pred_time": meta_test.iloc[i]["pred_time"],
        "num_pings": meta_test.iloc[i]["num_pings"],
        "input_dt": float(meta_test.iloc[i]["input_dt"]),
        "target_dt": float(meta_test.iloc[i]["target_dt"]),
        "delta_t_from_last_x": float(X_test_unscaled[i, -1, dt_idx]),
        "dr_horizon_dt": float(dr_dt[i]),
        "last_speed": float(meta_test.iloc[i]["last_speed"]),
        "last_cog": float(meta_test.iloc[i]["last_cog"]),
        "last_heading": float(meta_test.iloc[i]["last_heading"]),
        "last_lat": float(last_pos_test[i, 0]),
        "last_lon": float(last_pos_test[i, 1]),
        "tcn_pred_dlat": float(pred_test[i, 0]),
        "tcn_pred_dlon": float(pred_test[i, 1]),
        "true_dlat": float(y_test[i, 0]),
        "true_dlon": float(y_test[i, 1]),
        "tcn_pred_lat": float(tcn_pred_latlon[i, 0]),
        "tcn_pred_lon": float(tcn_pred_latlon[i, 1]),
        "dr_pred_lat": float(dr_pred_latlon[i, 0]),
        "dr_pred_lon": float(dr_pred_latlon[i, 1]),
        "true_lat": float(true_latlon[i, 0]),
        "true_lon": float(true_latlon[i, 1]),
        "tcn_error_m": float(tcn_error_m[i]),
        "dr_error_m": float(dr_error_m[i]),
        "tcn_error_yds": tcn_err,
        "dr_error_yds": dr_err,
        "improvement_yds": improvement,
        "abs_improvement_yds": abs(improvement),
        "winner": winner,
    })

results_df = pd.DataFrame(results_rows)

tcn_summary = summarize_error(results_df["tcn_error_yds"].to_numpy())
dr_summary = summarize_error(results_df["dr_error_yds"].to_numpy())

print("\n=== FINAL TEST METRICS (YARDS) ===")
print(
    f"TCN | RMSE={tcn_summary['rmse_yds']:.2f} | "
    f"MAE={tcn_summary['mae_yds']:.2f} | "
    f"Median={tcn_summary['median_yds']:.2f} | "
    f"P95={tcn_summary['p95_yds']:.2f}"
)
print(
    f"DR  | RMSE={dr_summary['rmse_yds']:.2f} | "
    f"MAE={dr_summary['mae_yds']:.2f} | "
    f"Median={dr_summary['median_yds']:.2f} | "
    f"P95={dr_summary['p95_yds']:.2f}"
)
print(f"\nTCN better on {(results_df['winner'] == 'TCN').mean() * 100:.2f}% of predictions")
print(f"DR better on {(results_df['winner'] == 'DR').mean() * 100:.2f}% of predictions")

results_path = ARTIFACTS_DIR / "results_df.csv"
model_path = ARTIFACTS_DIR / "tcn_model.pt"
scaler_path = ARTIFACTS_DIR / "feature_scaler.pkl"
metrics_path = ARTIFACTS_DIR / "metrics_summary.json"
config_path = ARTIFACTS_DIR / "config.json"

results_df.to_csv(results_path, index=False)
torch.save(model.state_dict(), model_path)
joblib.dump(feature_scaler, scaler_path)

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump({
        "tcn_summary_yds": tcn_summary,
        "dr_summary_yds": dr_summary,
        "tcn_win_percent": float((results_df["winner"] == "TCN").mean() * 100),
        "dr_win_percent": float((results_df["winner"] == "DR").mean() * 100),
    }, f, indent=2)

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(asdict(CFG), f, indent=2, default=str)

RUN_ARTIFACTS = {
    "base_dir": BASE_DIR,
    "artifacts_dir": ARTIFACTS_DIR,
    "results_path": results_path,
    "model_path": model_path,
    "scaler_path": scaler_path,
    "metrics_path": metrics_path,
    "config_path": config_path,
    "test_path": BASE_DIR / CFG.test_path,
    "train_path": BASE_DIR / CFG.train_path,
    "results_df": results_df,
    "test_df_raw": test_df_raw,
    "model": model,
    "feature_scaler": feature_scaler,
    "config": CFG,
}

print("\nSaved artifacts:")
for k in ["artifacts_dir", "results_path", "model_path", "scaler_path", "metrics_path", "config_path"]:
    print(f"{k}: {RUN_ARTIFACTS[k]}")

display(results_df.head(10))

Device: cuda
Epoch 001/100 | train_loss=0.00453143
Epoch 002/100 | train_loss=0.00172774
Epoch 003/100 | train_loss=0.00110534
Epoch 004/100 | train_loss=0.00081263
Epoch 005/100 | train_loss=0.00063236
Epoch 006/100 | train_loss=0.00052324
Epoch 007/100 | train_loss=0.00044513
Epoch 008/100 | train_loss=0.00038280
Epoch 009/100 | train_loss=0.00033268
Epoch 010/100 | train_loss=0.00029719
Epoch 011/100 | train_loss=0.00026700
Epoch 012/100 | train_loss=0.00023356
Epoch 013/100 | train_loss=0.00021561
Epoch 014/100 | train_loss=0.00020625
Epoch 015/100 | train_loss=0.00019032
Epoch 016/100 | train_loss=0.00017728
Epoch 017/100 | train_loss=0.00016179
Epoch 018/100 | train_loss=0.00015528
Epoch 019/100 | train_loss=0.00014323
Epoch 020/100 | train_loss=0.00013860
Epoch 021/100 | train_loss=0.00013482
Epoch 022/100 | train_loss=0.00012219
Epoch 023/100 | train_loss=0.00012370
Epoch 024/100 | train_loss=0.00011645
Epoch 025/100 | train_loss=0.00011158
Epoch 026/100 | train_loss=0.00011023

,idx,row_id,MMSI,voyage_id,anchor_time,pred_time,num_pings,input_dt,target_dt,delta_t_from_last_x,...,dr_pred_lon,true_lat,true_lon,tcn_error_m,dr_error_m,tcn_error_yds,dr_error_yds,improvement_yds,abs_improvement_yds,winner
0,0,10_209425000_24,209425000,10_209425000,2024-03-03 12:50:00,2024-03-03 12:55:00,50,300.0,300.0,300.0,...,-77.327087,21.997440,-77.326630,72.641335,83.838539,79.441528,91.686935,12.245407,12.245407,TCN
1,1,10_209425000_25,209425000,10_209425000,2024-03-03 12:55:00,2024-03-03 13:00:00,50,300.0,300.0,300.0,...,-77.307678,21.980537,-77.305977,261.644684,248.152756,286.138092,271.383148,-14.754944,14.754944,DR
2,2,10_209425000_26,209425000,10_209425000,2024-03-03 13:00:00,2024-03-03 13:05:00,50,300.0,300.0,300.0,...,-77.287392,21.966970,-77.287994,248.407547,180.237411,271.661804,197.110031,-74.551773,74.551773,DR
3,3,10_209425000_27,209425000,10_209425000,2024-03-03 13:05:00,2024-03-03 13:10:00,50,300.0,300.0,300.0,...,-77.269157,21.953373,-77.269913,206.346176,98.936623,225.662918,108.198402,-117.464516,117.464516,DR
4,4,10_209425000_28,209425000,10_209425000,2024-03-03 13:10:00,2024-03-03 13:15:00,50,300.0,300.0,300.0,...,-77.251122,21.938993,-77.250427,155.182159,73.497658,169.709274,80.378014,-89.331261,89.331261,DR
5,5,10_209425000_29,209425000,10_209425000,2024-03-03 13:15:00,2024-03-03 13:20:00,50,300.0,300.0,300.0,...,-77.231461,21.924856,-77.230965,157.647598,53.628414,172.405502,58.648746,-113.756756,113.756756,DR
6,6,10_209425000_30,209425000,10_209425000,2024-03-03 13:20:00,2024-03-03 13:25:00,50,300.0,300.0,300.0,...,-77.212074,21.911028,-77.211922,167.988724,16.792149,183.714691,18.364117,-165.350574,165.350574,DR
7,7,10_209425000_31,209425000,10_209425000,2024-03-03 13:25:00,2024-03-03 13:30:00,50,300.0,300.0,300.0,...,-77.192558,21.897043,-77.192978,115.230659,65.889290,126.017776,72.057404,-53.960373,53.960373,DR
8,8,10_209425000_32,209425000,10_209425000,2024-03-03 13:30:00,2024-03-03 13:35:00,50,300.0,300.0,300.0,...,-77.173615,21.883987,-77.175690,292.135986,227.436264,319.483795,248.727310,-70.756485,70.756485,DR
9,9,10_209425000_33,209425000,10_209425000,2024-03-03 13:35:00,2024-03-03 13:40:00,50,300.0,300.0,300.0,...,-77.156815,21.869675,-77.156723,76.676750,23.615707,83.854713,25.826450,-58.028263,58.028263,DR


In [2]:
from pathlib import Path
import pandas as pd
import folium
from IPython.display import display


# ============================================================
# EASY PATH ACCESS
# ============================================================

RESULTS_PATH = RUN_ARTIFACTS["results_path"]
TEST_PATH = RUN_ARTIFACTS["test_path"]
MAP_OUTPUT_DIR = RUN_ARTIFACTS["artifacts_dir"] / "indexed_maps"
MAP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# MAP HELPERS
# ============================================================

TIME_COL = "TIME"
LAT_COL = "LAT"
LON_COL = "LON"
MMSI_COL = "MMSI"
VOYAGE_COL = "voyage_id"


def load_results(results_path):
    df = pd.read_csv(results_path)
    for col in ["anchor_time", "pred_time", "TIME"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    df[MMSI_COL] = df[MMSI_COL].astype(str)
    df[VOYAGE_COL] = df[VOYAGE_COL].astype(str)
    return df


def load_test_data(test_path):
    df = pd.read_csv(test_path)
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df[MMSI_COL] = df[MMSI_COL].astype(str)
    df[VOYAGE_COL] = df[VOYAGE_COL].astype(str)
    return df.sort_values([VOYAGE_COL, TIME_COL]).reset_index(drop=True)


def resolve_row(results_df, idx=None, row_id=None):
    if row_id is not None:
        hits = results_df[results_df["row_id"].astype(str) == str(row_id)]
        if hits.empty:
            raise ValueError(f"row_id={row_id} not found")
        return hits.iloc[0]

    if idx is None:
        idx = 0

    hits = results_df[results_df["idx"] == idx]
    if not hits.empty:
        return hits.iloc[0]

    if idx < 0 or idx >= len(results_df):
        raise IndexError(f"idx={idx} out of bounds for results_df length {len(results_df)}")

    return results_df.iloc[idx]


def get_prior_track(test_df, row, max_previous_points=200):
    mmsi = str(row[MMSI_COL])
    voyage_id = str(row[VOYAGE_COL])
    anchor_time = row.get("anchor_time")

    voyage_df = test_df[
        (test_df[MMSI_COL] == mmsi) &
        (test_df[VOYAGE_COL] == voyage_id)
    ].copy()

    if voyage_df.empty:
        raise ValueError(f"No matching voyage found for MMSI={mmsi}, voyage_id={voyage_id}")

    voyage_df = voyage_df.sort_values(TIME_COL).reset_index(drop=True)
    prior_df = voyage_df[voyage_df[TIME_COL] <= anchor_time].copy()

    if prior_df.empty:
        prior_df = voyage_df.iloc[[0]].copy()

    if max_previous_points is not None and len(prior_df) > max_previous_points:
        prior_df = prior_df.iloc[-max_previous_points:].copy()

    return prior_df.reset_index(drop=True)


def format_popup(lines):
    return "<br>".join(lines)


def add_text_label(target_map, lat, lon, text, color, dx=10, dy=-12):
    html = f"""
    <div style="
        position: relative;
        transform: translate({dx}px, {dy}px);
        font-size: 12px;
        font-weight: 700;
        color: {color};
        background-color: rgba(255,255,255,0.85);
        border: 1px solid {color};
        border-radius: 4px;
        padding: 1px 4px;
        white-space: nowrap;
        display: inline-block;
    ">
        {text}
    </div>
    """
    folium.Marker(
        [lat, lon],
        icon=folium.DivIcon(
            icon_size=(100, 20),
            icon_anchor=(0, 0),
            html=html
        )
    ).add_to(target_map)


def add_prior_track(target_map, track_df):
    pts = track_df[[LAT_COL, LON_COL]].dropna().values.tolist()

    if len(pts) >= 2:
        folium.PolyLine(
            pts,
            color="gray",
            weight=3,
            opacity=0.8,
            tooltip="Previous trajectory"
        ).add_to(target_map)

    for i, (_, r) in enumerate(track_df.iterrows()):
        folium.CircleMarker(
            location=[r[LAT_COL], r[LON_COL]],
            radius=2,
            color="gray",
            fill=True,
            fill_color="gray",
            fill_opacity=0.75,
            opacity=0.75,
            popup=format_popup([
                f"track_idx: {i}",
                f"TIME: {r[TIME_COL]}",
                f"LAT: {r[LAT_COL]:.6f}",
                f"LON: {r[LON_COL]:.6f}",
            ]),
        ).add_to(target_map)


def add_anchor_and_truth(target_map, row):
    last_lat = float(row["last_lat"])
    last_lon = float(row["last_lon"])
    true_lat = float(row["true_lat"])
    true_lon = float(row["true_lon"])

    folium.Marker(
        [last_lat, last_lon],
        tooltip="Anchor / current point",
        popup=format_popup([
            "Anchor / current point",
            f"anchor_time: {row.get('anchor_time', 'NA')}",
            f"LAT: {last_lat:.6f}",
            f"LON: {last_lon:.6f}",
            f"SOG: {row.get('last_speed', 'NA')}",
            f"COG: {row.get('last_cog', 'NA')}",
        ]),
        icon=folium.Icon(color="black", icon="play", prefix="fa"),
    ).add_to(target_map)
    add_text_label(target_map, last_lat, last_lon, "ANCHOR", "black", dx=12, dy=-10)

    folium.Marker(
        [true_lat, true_lon],
        tooltip="True next point",
        popup=format_popup([
            "True next point",
            f"pred_time: {row.get('pred_time', 'NA')}",
            f"LAT: {true_lat:.6f}",
            f"LON: {true_lon:.6f}",
        ]),
        icon=folium.Icon(color="green", icon="ok", prefix="fa"),
    ).add_to(target_map)
    add_text_label(target_map, true_lat, true_lon, "TRUE", "green", dx=12, dy=-10)

    folium.PolyLine(
        [[last_lat, last_lon], [true_lat, true_lon]],
        color="green",
        weight=3,
        opacity=0.9,
        dash_array="8,6",
        tooltip="Actual movement",
    ).add_to(target_map)


def add_tcn_prediction(target_map, row):
    last_lat = float(row["last_lat"])
    last_lon = float(row["last_lon"])
    pred_lat = float(row["tcn_pred_lat"])
    pred_lon = float(row["tcn_pred_lon"])
    err = float(row["tcn_error_yds"])

    folium.Marker(
        [pred_lat, pred_lon],
        tooltip="TCN prediction",
        popup=format_popup([
            "TCN prediction",
            f"LAT: {pred_lat:.6f}",
            f"LON: {pred_lon:.6f}",
            f"TCN error: {err:.2f} yds",
        ]),
        icon=folium.Icon(color="blue", icon="info-sign"),
    ).add_to(target_map)
    add_text_label(target_map, pred_lat, pred_lon, "TCN", "blue", dx=12, dy=-10)

    folium.PolyLine(
        [[last_lat, last_lon], [pred_lat, pred_lon]],
        color="blue",
        weight=3,
        opacity=0.9,
        dash_array="4,6",
        tooltip=f"TCN predicted movement | error={err:.2f} yds",
    ).add_to(target_map)


def add_dr_prediction(target_map, row):
    last_lat = float(row["last_lat"])
    last_lon = float(row["last_lon"])
    pred_lat = float(row["dr_pred_lat"])
    pred_lon = float(row["dr_pred_lon"])
    err = float(row["dr_error_yds"])

    folium.Marker(
        [pred_lat, pred_lon],
        tooltip="Dead reckoning prediction",
        popup=format_popup([
            "Dead reckoning prediction",
            f"LAT: {pred_lat:.6f}",
            f"LON: {pred_lon:.6f}",
            f"DR error: {err:.2f} yds",
        ]),
        icon=folium.Icon(color="red", icon="flag"),
    ).add_to(target_map)
    add_text_label(target_map, pred_lat, pred_lon, "DR", "red", dx=12, dy=-10)

    folium.PolyLine(
        [[last_lat, last_lon], [pred_lat, pred_lon]],
        color="red",
        weight=3,
        opacity=0.9,
        dash_array="4,6",
        tooltip=f"DR predicted movement | error={err:.2f} yds",
    ).add_to(target_map)


def add_header_and_legend(target_map, row):
    winner = row.get("winner", "NA")
    html = f"""
    <div style="
        position: fixed;
        top: 10px;
        left: 50px;
        z-index: 9999;
        background-color: white;
        border: 2px solid #444;
        border-radius: 8px;
        padding: 10px 12px;
        font-size: 13px;
        line-height: 1.35;
        box-shadow: 0 2px 8px rgba(0,0,0,0.25);
        max-width: 620px;
    ">
        <b>Indexed Result Comparison</b><br>
        idx: {row.get('idx', 'NA')} | row_id: {row.get('row_id', 'NA')}<br>
        MMSI: {row.get('MMSI', 'NA')} | voyage_id: {row.get('voyage_id', 'NA')}<br>
        anchor_time: {row.get('anchor_time', 'NA')}<br>
        pred_time: {row.get('pred_time', 'NA')}<br>
        TCN error: {float(row.get('tcn_error_yds', float('nan'))):.2f} yds |
        DR error: {float(row.get('dr_error_yds', float('nan'))):.2f} yds |
        Improvement: {float(row.get('improvement_yds', float('nan'))):.2f} yds<br>
        Winner: <b>{winner}</b><br><br>
        <b>Legend</b><br>
        Gray = previous trajectory<br>
        Black = anchor / current point<br>
        Green = true next point<br>
        Blue = TCN prediction<br>
        Red = DR prediction
    </div>
    """
    target_map.get_root().html.add_child(folium.Element(html))


def build_single_map(row, prior_df):
    center_lat = float(row["true_lat"])
    center_lon = float(row["true_lon"])

    m = folium.Map(location=[center_lat, center_lon], zoom_start=11, control_scale=True)

    add_prior_track(m, prior_df)
    add_anchor_and_truth(m, row)
    add_tcn_prediction(m, row)
    add_dr_prediction(m, row)
    add_header_and_legend(m, row)

    return m


def show_indexed_result(idx=None, row_id=None, max_previous_points=200, save_html=True):
    results_df = load_results(RESULTS_PATH)
    test_df = load_test_data(TEST_PATH)

    row = resolve_row(results_df, idx=idx, row_id=row_id)
    prior_df = get_prior_track(test_df, row, max_previous_points=max_previous_points)
    m = build_single_map(row, prior_df)

    if save_html:
        safe_idx = str(row.get("idx", "NA"))
        safe_row_id = str(row.get("row_id", "NA")).replace("/", "_")
        out_path = MAP_OUTPUT_DIR / f"singlemap_result_idx_{safe_idx}_rowid_{safe_row_id}.html"
        m.save(str(out_path))
        print(f"Saved map to: {out_path}")

    print(
        f"idx={row['idx']} | row_id={row['row_id']} | "
        f"TCN={row['tcn_error_yds']:.2f} yds | "
        f"DR={row['dr_error_yds']:.2f} yds | "
        f"Winner={row['winner']}"
    )

    display(m)
    return m


def show_tcn_win(rank=0, sort_by="improvement_yds", ascending=False, max_previous_points=200):
    results_df = load_results(RESULTS_PATH)
    wins = results_df[results_df["winner"] == "TCN"].copy()

    if sort_by is not None:
        wins = wins.sort_values(sort_by, ascending=ascending).reset_index(drop=True)

    if rank < 0 or rank >= len(wins):
        raise IndexError(f"rank={rank} out of bounds for {len(wins)} TCN wins")

    row = wins.iloc[rank]
    print(
        f"Showing TCN win rank {rank} | idx={row['idx']} | "
        f"TCN={row['tcn_error_yds']:.2f} yds | "
        f"DR={row['dr_error_yds']:.2f} yds | "
        f"Improvement={row['improvement_yds']:.2f} yds"
    )
    return show_indexed_result(idx=int(row["idx"]), max_previous_points=max_previous_points)
def show_tcn_win(rank=0, sort_by="improvement_yds", ascending=False, max_previous_points=200):
    results_df = load_results(RESULTS_PATH)
    wins = results_df[results_df["winner"] == "TCN"].copy()

    if sort_by is not None:
        wins = wins.sort_values(sort_by, ascending=ascending).reset_index(drop=True)

    if rank < 0 or rank >= len(wins):
        raise IndexError(f"rank={rank} out of bounds for {len(wins)} TCN wins")

    row = wins.iloc[rank]
    print(
        f"Showing TCN win rank {rank} | idx={row['idx']} | "
        f"TCN={row['tcn_error_yds']:.2f} yds | "
        f"DR={row['dr_error_yds']:.2f} yds | "
        f"Improvement={row['improvement_yds']:.2f} yds"
    )
    return show_indexed_result(idx=int(row["idx"]), max_previous_points=max_previous_points)


# Examples:
# show_indexed_result(idx=0)
# show_indexed_result(idx=25, max_previous_points=100)
# show_indexed_result(row_id="123456")

In [16]:
show_tcn_win(rank=82)

Showing TCN win rank 82 | idx=929 | TCN=351.81 yds | DR=1433.35 yds | Improvement=1081.54 yds
Saved map to: /home/jacobhardy/capstone/notebooks/latest_tcn_run/indexed_maps/singlemap_result_idx_929_rowid_1_357211000_34.html
idx=929 | row_id=1_357211000_34 | TCN=351.81 yds | DR=1433.35 yds | Winner=TCN
